##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [ ]:
#pip install pinecone

From 'pinecone' library import Pinecone, ServerlessSpec.

In [7]:
from pinecone import Pinecone, ServerlessSpec

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [14]:
import os
import time
from dotenv import load_dotenv, find_dotenv

# load the key from the .env file
load_dotenv(find_dotenv())
api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=api_key)

# create the index with a built-in embedding model, so Pinecone embeds the text for us
index_name = "my-quickstart-index"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"}
        }
    )

index = pc.Index(index_name)

# my own text strings
records = [
    {"_id": "rec1", "chunk_text": "Cycling to work saves money on fuel and keeps you fit.", "category": "lifestyle"},
    {"_id": "rec2", "chunk_text": "The city plans to build more bicycle lanes downtown.", "category": "city"},
    {"_id": "rec3", "chunk_text": "Tomato pasta needs ripe tomatoes, garlic and fresh basil.", "category": "food"},
    {"_id": "rec4", "chunk_text": "I cooked a big pot of vegetable soup on the weekend.", "category": "food"},
    {"_id": "rec5", "chunk_text": "The football team won the championship after a late goal.", "category": "sport"},
    {"_id": "rec6", "chunk_text": "Python has simple syntax and many libraries for data science.", "category": "tech"},
    {"_id": "rec7", "chunk_text": "Vector databases help you search documents by meaning.", "category": "tech"},
]

# this SDK wants keyword arguments here
index.upsert_records(namespace="my-namespace", records=records)

# upserted data takes a few seconds to become searchable
time.sleep(10)

# search with my own text
results = index.search(
    namespace="my-namespace",
    query={
        "top_k": 3,
        "inputs": {"text": "What should I cook for dinner?"}
    }
)

Print search results.

In [18]:
for hit in results.result.hits:
    print(f"id: {hit.id:<6} | score: {round(hit.score, 3):<6} | text: {hit.fields['chunk_text']}")


id: rec4   | score: 0.193  | text: I cooked a big pot of vegetable soup on the weekend.
id: rec3   | score: 0.113  | text: Tomato pasta needs ripe tomatoes, garlic and fresh basil.
id: rec1   | score: 0.034  | text: Cycling to work saves money on fuel and keeps you fit.


**My Observation:**
- I stored  7 sentences in Pinecone. The Pinecone turned the text into vectors.
- I searched "What should I cook for dinner?" and the top 3 were the soup sentence (0.193), the pasta sentence (0.113) and the cycling sentence (0.034).
- The two food sentences came first even though they share no words with my query, so the search works by meaning, not keywords.
